In [1]:
import os

if os.path.basename(os.getcwd()) != "notebooks":
    os.chdir("/home/jovyan/notebooks")

print("cwd:", os.getcwd())

cwd: /home/jovyan/notebooks


# ClimaCity Paris -- Jour 2
## Spark SQL, Delta Lake et Structured Streaming

**Module** : Traitement de données massives avec Apache Spark et PySpark  
**Durée** : 1 journée (6 heures effectives)  
**Prérequis** : Avoir complété le Jour 1 -- la table `disponibilite_consolidee.parquet` doit être présente dans `data/output/`

---

Ce notebook couvre l'intégralité du Jour 2 du projet ClimaCity Paris.  
Il se divise en deux grandes parties :

- **Partie 1 -- Matin (3 h)** : Spark SQL et l'API de fenêtrage analytique, puis Delta Lake
  pour la persistance transactionnelle (écriture, time-travel, `MERGE INTO`).
- **Partie 2 -- Après-midi (3 h)** : Structured Streaming -- connexion à un flux simulé
  de mises à jour de stations, agrégations sur fenêtres glissantes, gestion des données
  tardives (late data) et déclenchement d'alertes.

> **Convention** : les cellules `# [EXERCICE]` contiennent une consigne à compléter.  
> Les cellules `# [CORRECTION]` proposent une solution -- ne les regardez qu'après avoir tenté.


---
> **Transparence sur l'usage d'un assistant IA**
>
> Plusieurs cellules de ce notebook (indiquées individuellement par un
> commentaire à la première personne, ex. *"J'ai adapté..."*, *"J'ai corrigé..."*)
> ont été complétées avec l'aide de l'assistant IA **Claude / Claude Code**
> (Anthropic), après que nous avons constaté que certaines cellules d'exercice
> reposaient sur des données ou variables absentes du projet réel (schéma
> différent du jeu de données réellement fourni, plage de dates 2022-2023 de
> l'énoncé alors que nos données couvrent 2020-2021, fichiers "pré-préparés"
> jamais générés par aucune cellule).
>
> Chaque correction a été vérifiée par rapport aux données réelles (schéma,
> plage de dates, comptages) avant d'être acceptée -- voir la section
> *« Transparence sur l'utilisation d'un assistant IA »* du `README.md` du
> projet pour le détail complet de la démarche et de ce qui reste de notre
> propre analyse (choix de conception, interprétation des résultats).
---


---
## Section 0 -- Configuration

Même structure de chemins qu'au Jour 1. La table consolidée produite hier est le
point de départ de toutes les analyses.


In [2]:
from pathlib import Path
import time

# ── Chemins ─────────────────────────────────────────────────────────────────
DATA_DIR           = Path("../data")
OUTPUT_DIR         = DATA_DIR / "output"
VELIB_CONSOLIDE    = OUTPUT_DIR / "disponibilite_consolidee.parquet"
DELTA_DISPONIBLE   = OUTPUT_DIR / "delta" / "disponibilite"
DELTA_ALERTES      = OUTPUT_DIR / "delta" / "alertes"
STREAM_SOURCE_DIR  = OUTPUT_DIR / "stream_input"    # répertoire surveillé par Spark
STREAM_CHECKPOINT  = OUTPUT_DIR / "checkpoints"

for p in [VELIB_CONSOLIDE]:
    assert p.exists(), f"Fichier manquant : {p} -- relancez le Jour 1"

for p in [DELTA_DISPONIBLE, DELTA_ALERTES, STREAM_SOURCE_DIR, STREAM_CHECKPOINT]:
    p.mkdir(parents=True, exist_ok=True)

# ── Paramètres ───────────────────────────────────────────────────────────────
APP_NAME      = "ClimaCity-Paris-Jour2"
SHUFFLE_PARTS = 8
SEED          = 42


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Delta Lake requiert le package delta-spark
# Vérifiez que votre environnement conda l'inclut avant de démarrer.
spark = (
    SparkSession.builder
    .appName(APP_NAME)
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", SHUFFLE_PARTS)
    .config("spark.driver.memory", "4g")
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("WARN")

print(f"Spark {spark.version} -- Delta Lake activé")
print(f"Spark UI : http://localhost:4040")


Spark 3.5.3 -- Delta Lake activé
Spark UI : http://localhost:4040


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 51282)
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/opt/conda/lib/python3.11/socketserver.py", line 755, in __init__
    self.handle()
  File "/usr/local/spark/python/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/usr/local/spark/python/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
                           ^^^^^^
  File "/usr/local/spark/python/pyspark/accumulators.py", line 271, in accum_updates
    num_updates =

In [4]:
# Chargement de la table consolidée produite au Jour 1
df = spark.read.parquet(str(VELIB_CONSOLIDE))
df.cache()
df.count()   # force la mise en cache

print(f"Table consolidée : {df.count():,} lignes  |  {len(df.columns)} colonnes")
df.printSchema()


Table consolidée : 10,921,968 lignes  |  18 colonnes
root
 |-- nom_station: string (nullable = true)
 |-- capacite: integer (nullable = true)
 |-- horodatage: timestamp (nullable = true)
 |-- velos_meca: integer (nullable = true)
 |-- velos_elec: integer (nullable = true)
 |-- velos_total: integer (nullable = true)
 |-- taux_occupation: double (nullable = true)
 |-- statut: string (nullable = true)
 |-- jour_sem: integer (nullable = true)
 |-- heure: integer (nullable = true)
 |-- is_weekend: boolean (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- humidite_pct: double (nullable = true)
 |-- vent_kmh: double (nullable = true)
 |-- precipitation_mm: double (nullable = true)
 |-- est_pluie: boolean (nullable = true)
 |-- annee: integer (nullable = true)
 |-- mois: integer (nullable = true)



---
# PARTIE 1 -- Spark SQL (matin)

## 1.1 Vues temporaires et premières requêtes SQL

L'API DataFrame et Spark SQL sont **entièrement interchangeables** : elles produisent
le même plan d'exécution physique après passage par le Catalyst optimizer.
Le choix entre les deux est une question de lisibilité et d'habitude.

La règle pratique : SQL excelle pour les agrégations complexes et le fenêtrage.
L'API DataFrame est plus commode pour les traitements programmatiques (boucles,
conditions dynamiques, chaînage de transformations).


In [5]:
import sys
print("Python :", sys.version)
print("Exécuté depuis :", sys.executable)

Python : 3.11.10 | packaged by conda-forge | (main, Oct 16 2024, 01:19:04) [GCC 13.3.0]
Exécuté depuis : /opt/conda/bin/python


In [6]:
# Enregistrement des vues temporaires
# Une vue temporaire n'existe que pour la durée de la session Spark.
# Elle ne copie pas les données -- c'est un alias sur le DataFrame.
df.createOrReplaceTempView("disponibilite")

# Vérification
spark.sql("SHOW VIEWS").show()


+---------+-------------+-----------+
|namespace|     viewName|isTemporary|
+---------+-------------+-----------+
|         |disponibilite|       true|
+---------+-------------+-----------+



In [7]:
df.columns

['nom_station',
 'capacite',
 'horodatage',
 'velos_meca',
 'velos_elec',
 'velos_total',
 'taux_occupation',
 'statut',
 'jour_sem',
 'heure',
 'is_weekend',
 'temperature_c',
 'humidite_pct',
 'vent_kmh',
 'precipitation_mm',
 'est_pluie',
 'annee',
 'mois']

In [8]:
# Première requête : distribution des statuts par arrondissement :
# Nom de 'snapshots', taux d'occuoation moyen, écarrt-type de l'occupation
spark.sql("""
SELECT 
    statut,
    COUNT(*) AS nb_snapshots,
    ROUND(AVG(taux_occupation), 4) AS taux_moyen,
    ROUND(STDDEV(taux_occupation), 4) AS taux_stddev
FROM disponibilite
GROUP BY statut
ORDER BY statut
""").show(40)


+------+------------+----------+-----------+
|statut|nb_snapshots|taux_moyen|taux_stddev|
+------+------------+----------+-----------+
|normal|     8618099|    0.4022|     0.2229|
| plein|      428471|    0.9497|     0.0305|
|  vide|     1875398|    0.0471|     0.0316|
+------+------------+----------+-----------+



In [9]:
# Les fonctions temporelles SQL sont disponibles directement
# Taux moyen d'occupation, nombre de snapsohts, nombre de stations par heure
spark.sql("""
SELECT 
    heure,
    ROUND(AVG(taux_occupation), 4) AS taux_moyen,
    COUNT(*) AS nb_snapshots,
    COUNT(DISTINCT nom_station) AS nb_stations
FROM disponibilite
GROUP BY heure
ORDER BY heure
""").show(24)


+-----+----------+------------+-----------+
|heure|taux_moyen|nb_snapshots|nb_stations|
+-----+----------+------------+-----------+
|    0|    0.3874|      197172|       1388|
|    1|    0.3887|      213818|       1388|
|    2|    0.3909|      413766|       1388|
|    3|     0.391|      483188|       1388|
|    4|    0.3902|      427661|       1388|
|    5|    0.3905|      522073|       1388|
|    6|     0.378|      440161|       1388|
|    7|    0.3607|      522072|       1388|
|    8|    0.3607|      454049|       1388|
|    9|    0.3615|      485967|       1388|
|   10|    0.3525|      512353|       1388|
|   11|    0.3391|      577627|       1388|
|   12|    0.3382|      387398|       1389|
|   13|    0.3401|      529013|       1389|
|   14|    0.3401|      552638|       1388|
|   15|    0.3347|      495704|       1388|
|   16|     0.327|      281868|       1389|
|   17|     0.337|      374887|       1388|
|   18|    0.3401|      402665|       1388|
|   19|    0.3508|      567889| 

---
## 1.2 Questions métier -- Requêtes analytiques

L'équipe métier a soumis trois questions auxquelles votre plateforme doit répondre.
Nous allons les traiter une par une avec Spark SQL.


### Question 1 : Ruptures en heure de pointe matinale

> Quelles sont les 10 stations les plus souvent en rupture totale (zéro vélo disponible,
> mécanique ou électrique) entre 7 h et 10 h, les jours de semaine,
> en excluant les jours fériés français ?

Les jours fériés français sont injectés comme une petite table de référence --
c'est l'occasion d'illustrer la `broadcast join` en SQL.


In [10]:
# Table de jours fériés (2022-2023) -- injectée en broadcast
# Source : légifrance.gouv.fr
jours_feries = spark.createDataFrame([
    ("2022-01-01",), ("2022-04-18",), ("2022-05-01",), ("2022-05-08",),
    ("2022-05-26",), ("2022-06-06",), ("2022-07-14",), ("2022-08-15",),
    ("2022-11-01",), ("2022-11-11",), ("2022-12-25",),
    ("2023-01-01",), ("2023-04-10",), ("2023-05-01",), ("2023-05-08",),
    ("2023-05-18",), ("2023-05-29",), ("2023-07-14",), ("2023-08-15",),
    ("2023-11-01",), ("2023-11-11",), ("2023-12-25",),
], ["date_ferie"])

jours_feries = jours_feries.withColumn(
    "date_ferie", F.to_date("date_ferie", "yyyy-MM-dd")
)
jours_feries.createOrReplaceTempView("jours_feries")

print(f"{jours_feries.count()} jours fériés enregistrés (2022-2023)")


22 jours fériés enregistrés (2022-2023)


In [11]:
# Identifier les 10 stations Vélib' les plus en rupture de stock pendant les heures de pointe matinales, en excluant les jours fériés.
# On ne s'intéresse qu'aux stations trè!s fréquentées,ayant plus de 100 observations (snapshots)
# -> Id de la station, noù de la station, arroondissement, nombre d'observations (snapshots)
df_q1 = spark.sql("""
SELECT 
    nom_station,
    COUNT(*) AS nb_snapshots_matin,
    SUM(CASE WHEN velos_total = 0 THEN 1 ELSE 0 END) AS nb_ruptures,
    ROUND(100.0 * SUM(CASE WHEN velos_total = 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_rupture
FROM disponibilite d
LEFT JOIN jours_feries jf 
    ON DATE(d.horodatage) = jf.date_ferie
WHERE 
    jf.date_ferie IS NULL          -- exclut les jours fériés
    AND heure BETWEEN 7 AND 9      -- 7h à 9h59 = heures de pointe matinales
    AND jour_sem BETWEEN 2 AND 6   -- lundi (2) à vendredi (6) = jours ouvrés
GROUP BY nom_station
HAVING COUNT(*) > 100              -- stations très fréquentées
ORDER BY pct_rupture DESC
LIMIT 10
""")

df_q1.show(truncate=False)

+------------------------------+------------------+-----------+-----------+
|nom_station                   |nb_snapshots_matin|nb_ruptures|pct_rupture|
+------------------------------+------------------+-----------+-----------+
|Manufacture Nationale         |761               |761        |100.00     |
|Edouard Vaillant - Galliéni   |761               |761        |100.00     |
|Parc Floral du Bel Air        |376               |376        |100.00     |
|Porte de Pantin - Petits Ponts|761               |761        |100.00     |
|8 Mai 1945 - 10 Juillet 1940  |761               |761        |100.00     |
|Gare RER les Ardoines         |761               |761        |100.00     |
|Macdonald - Césaria Evora     |761               |761        |100.00     |
|Carnot - Général Lanrezac     |761               |705        |92.64      |
|Mairie du 20ème               |761               |702        |92.25      |
|Vanne - Général de Gaulle     |761               |565        |74.24      |
+-----------

### Question 2 : Impact de la pluie sur le taux d'occupation

> La pluie réduit-elle statistiquement le taux d'occupation moyen du réseau ?
> De combien de points en moyenne ? L'effet est-il homogène selon les arrondissements ?


In [12]:
# Distribution statistique par quartiles du taux d'occupation en fonction de la météo
df_q2 = spark.sql("""
SELECT
    est_pluie,
    COUNT(*) AS nb_snapshots,
    ROUND(AVG(taux_occupation), 4) AS taux_moyen,
    ROUND(percentile(taux_occupation, 0.25), 4) AS q1,
    ROUND(percentile(taux_occupation, 0.50), 4) AS mediane,
    ROUND(percentile(taux_occupation, 0.75), 4) AS q3
FROM disponibilite
WHERE est_pluie IS NOT NULL
GROUP BY est_pluie
ORDER BY est_pluie
""")
df_q2.show()


+---------+------------+----------+------+-------+------+
|est_pluie|nb_snapshots|taux_moyen|    q1|mediane|    q3|
+---------+------------+----------+------+-------+------+
|    false|    10423394|    0.3621|0.1389|    0.3|  0.55|
|     true|      498574|    0.3752|0.1528|   0.32|0.5667|
+---------+------------+----------+------+-------+------+



In [13]:
# Effet de la pluie par arrondissement
df_q2_arr = spark.sql("""
SELECT
    nom_station,
    ROUND(AVG(CASE WHEN est_pluie = false THEN taux_occupation END), 4) AS taux_sec,
    ROUND(AVG(CASE WHEN est_pluie = true  THEN taux_occupation END), 4) AS taux_pluie,
    ROUND(
        AVG(CASE WHEN est_pluie = true  THEN taux_occupation END) 
      - AVG(CASE WHEN est_pluie = false THEN taux_occupation END),
        4
    ) AS delta
FROM disponibilite
WHERE est_pluie IS NOT NULL
GROUP BY nom_station
HAVING 
    COUNT(CASE WHEN est_pluie = false THEN 1 END) > 50   -- au moins 50 snapshots secs
    AND COUNT(CASE WHEN est_pluie = true  THEN 1 END) > 50 -- au moins 50 snapshots pluvieux
ORDER BY delta DESC
LIMIT 30
""")
df_q2_arr.show(30)
# Un delta négatif signifie que le taux d'occupation baisse sous la pluie
# (moins de vélos empruntés -> plus de vélos disponibles -> moins de bornettes libres)


+--------------------+--------+----------+------+
|         nom_station|taux_sec|taux_pluie| delta|
+--------------------+--------+----------+------+
|Route de Sèvres -...|  0.5799|    0.7907|0.2109|
|      Mahatma Gandhi|  0.4485|    0.6516|0.2031|
|Commandant Charco...|  0.5613|    0.7349|0.1736|
|Verdun - Place He...|  0.5561|    0.7293|0.1732|
|Route de la Muett...|  0.4808|    0.6485|0.1677|
|Hôtel de Ville de...|   0.373|    0.5369|0.1638|
|Place Charles de ...|  0.3731|    0.5265|0.1534|
|Taitbout - La Fay...|  0.4375|    0.5856|0.1481|
|Gare de Lyon - Va...|    0.52|    0.6643|0.1444|
|Gare de Lyon - Ro...|  0.4334|    0.5694| 0.136|
|Gustave Eiffel - ...|  0.3722|    0.5066|0.1344|
| Mairie de Montreuil|  0.2857|     0.417|0.1313|
|Charles de Gaulle...|  0.4228|    0.5499|0.1271|
|    Temple - Charlot|  0.5869|    0.7125|0.1256|
|  Gare de Courbevoie|   0.302|     0.427|0.1251|
|Lucien Lanternier...|  0.4518|    0.5753|0.1234|
|Lobau - Hôtel de ...|  0.5341|    0.6558|0.1217|


### Question 3 : Saisonnalité intra-journalière

> Quelle station présente la plus forte amplitude entre son heure creuse et son heure
> de pointe au cours d'une journée type (taux_max - taux_min par heure) ?

C'est un cas d'usage typique des **fonctions de fenêtrage**.


In [14]:
# Quelles sont les 15 stations dont le comportement est le plus pendulaire — presque vides à certaines heures, saturées à d'autres ?
df_q3 = spark.sql("""
WITH profil_horaire AS (
    -- Pour chaque (station, heure), le taux moyen d'occupation
    SELECT 
        nom_station,
        heure,
        AVG(taux_occupation) AS taux_horaire
    FROM disponibilite
    GROUP BY nom_station, heure
)
SELECT 
    nom_station,
    ROUND(MIN(taux_horaire), 4) AS taux_min_journee,
    ROUND(MAX(taux_horaire), 4) AS taux_max_journee,
    ROUND(MAX(taux_horaire) - MIN(taux_horaire), 4) AS amplitude
FROM profil_horaire
GROUP BY nom_station
HAVING COUNT(*) = 24               -- station observée à toutes les heures
ORDER BY amplitude DESC
LIMIT 15
""")
df_q3.show(truncate=False)


+------------------------------------+----------------+----------------+---------+
|nom_station                         |taux_min_journee|taux_max_journee|amplitude|
+------------------------------------+----------------+----------------+---------+
|Gare du Nord - Denain               |0.1291          |0.8589          |0.7298   |
|Gare du Nord - Faubourg Saint-Denis |0.1347          |0.8204          |0.6858   |
|Léon - Doudeauville                 |0.1357          |0.7406          |0.6049   |
|Godot de Mauroy - Madeleine         |0.258           |0.8612          |0.6032   |
|Place de la Madeleine - Royale      |0.2775          |0.8725          |0.5951   |
|Danielle Casanova - Place Vendôme   |0.2121          |0.8013          |0.5892   |
|Erasme - Ulm                        |0.0803          |0.6561          |0.5758   |
|Riquet - Marx Dormoy                |0.1751          |0.7412          |0.5661   |
|Méchain - Faubourg Saint-Jacques    |0.1099          |0.6586          |0.5487   |
|Cam

In [15]:
# [EXERCICE]
# En utilisant Spark SQL, calculez pour chaque station :
# - le taux d'occupation moyen un jour de semaine sec (est_pluie = false)
# - le taux d'occupation moyen un week-end pluvieux (est_pluie = true)
# - le ratio entre les deux
# Affichez les 10 stations avec le ratio le plus élevé (plus forte différence).
#
# Rappel : est_weekend est un booléen, est_pluie aussi.
# ──────────────────────────────────────────────────────────────────────────

# Votre requête ici :
spark.sql("""
SELECT
    nom_station,
    ROUND(AVG(CASE 
        WHEN is_weekend = false AND est_pluie = false THEN taux_occupation 
    END), 4) AS taux_semaine_sec,
    ROUND(AVG(CASE 
        WHEN is_weekend = true AND est_pluie = true THEN taux_occupation 
    END), 4) AS taux_weekend_pluie,
    ROUND(
        AVG(CASE WHEN is_weekend = true AND est_pluie = true THEN taux_occupation END)
      / NULLIF(AVG(CASE WHEN is_weekend = false AND est_pluie = false THEN taux_occupation END), 0),
        4
    ) AS ratio
FROM disponibilite
WHERE est_pluie IS NOT NULL
GROUP BY nom_station
HAVING 
    COUNT(CASE WHEN is_weekend = false AND est_pluie = false THEN 1 END) > 50
    AND COUNT(CASE WHEN is_weekend = true AND est_pluie = true THEN 1 END) > 20
ORDER BY ratio DESC
LIMIT 10
""").show()


+--------------------+----------------+------------------+------+
|         nom_station|taux_semaine_sec|taux_weekend_pluie| ratio|
+--------------------+----------------+------------------+------+
|Place de la Divis...|          0.0935|            0.3188|3.4111|
|Cimetière de Mont...|          0.1383|            0.3872|2.8003|
|Carnot - Général ...|          0.0026|            0.0065|2.5545|
|Edouard Pailleron...|          0.2214|            0.5425|2.4504|
|Gare Montparnasse...|          0.1496|            0.3583|2.3953|
|Pierre et Marie C...|          0.1383|            0.3163|2.2871|
|Saint-Fargeau - M...|          0.1368|            0.3127|2.2859|
|Square Louise Michel|          0.1497|             0.315| 2.104|
|Gare Montparnasse...|          0.3145|            0.6477|2.0594|
|Frères Flavien - ...|          0.1421|            0.2895|2.0371|
+--------------------+----------------+------------------+------+



---
## 1.3 Fonctions de fenêtrage analytique

Les fonctions de fenêtrage (`WINDOW` / `OVER`) permettent de calculer des agrégats
**sans réduire le nombre de lignes** -- contrairement à `GROUP BY`.
Elles sont indispensables pour les analyses de séries temporelles.

### Les trois familles de fonctions fenêtrées

```
Ranking    : ROW_NUMBER, RANK, DENSE_RANK, NTILE
Navigation : LAG, LEAD, FIRST_VALUE, LAST_VALUE, NTH_VALUE
Agrégation : SUM, AVG, MIN, MAX, COUNT (avec clause OVER)
```


In [16]:
# Cas concret : pour chaque station, calculer le taux d'occupation
# de la fenêtre précédente (LAG) et suivante (LEAD),
# ainsi qu'une moyenne mobile sur 3 snapshots.

fenetre_station = (
    Window
    .partitionBy("nom_station")     # une fenêtre par station
    .orderBy("horodatage")          # ordonnée chronologiquement
)

df_avec_lag = (
    df
    .withColumn("taux_precedent", F.lag("taux_occupation").over(fenetre_station))
    .withColumn("taux_suivant",   F.lead("taux_occupation").over(fenetre_station))
    .withColumn(
        "moyenne_mobile_3",
        F.avg("taux_occupation").over(
            fenetre_station.rowsBetween(-1, 1)   # 1 avant, courant, 1 après
        )
    )
    .select("nom_station", "horodatage", "taux_occupation",
            "taux_precedent", "taux_suivant", "moyenne_mobile_3")
    .orderBy("nom_station", "horodatage")
)

df_avec_lag.show(10, truncate=False)


+-------------------------------------+-------------------+---------------+--------------+------------+--------------------+
|nom_station                          |horodatage         |taux_occupation|taux_precedent|taux_suivant|moyenne_mobile_3    |
+-------------------------------------+-------------------+---------------+--------------+------------+--------------------+
| Jean Bleuzen - Square du 11 Novembre|2020-11-26 12:59:00|0.0816         |NULL          |0.0816      |0.0816              |
| Jean Bleuzen - Square du 11 Novembre|2020-11-26 13:06:00|0.0816         |0.0816        |0.0816      |0.0816              |
| Jean Bleuzen - Square du 11 Novembre|2020-11-26 13:21:00|0.0816         |0.0816        |0.0816      |0.0816              |
| Jean Bleuzen - Square du 11 Novembre|2020-11-26 13:32:00|0.0816         |0.0816        |0.0612      |0.0748              |
| Jean Bleuzen - Square du 11 Novembre|2020-11-26 13:47:00|0.0612         |0.0816        |0.0408      |0.061200000000000004|


In [17]:
# Classement des stations par taux d'occupation moyen, à chaque heure de la journée
# ROW_NUMBER() numérote les lignes dans chaque partition (ici : chaque heure)

# Étape 1 : agrégat taux moyen par (station, heure)
df_taux_horaire = (
    df
    .groupBy("nom_station", "heure")
    .agg(F.avg("taux_occupation").alias("taux_moyen"))
)

# Étape 2 : fenêtre — partition par heure, ordre par taux moyen décroissant
fenetre_heure = (
    Window
    .partitionBy("heure")
    .orderBy(F.col("taux_moyen").desc())
)

# Étape 3 : appliquer ROW_NUMBER puis filtrer top 2 par heure
df_rank = (
    df_taux_horaire
    .withColumn("rang", F.row_number().over(fenetre_heure))
    .filter(F.col("rang") <= 2)
    .orderBy("heure", "rang")
    .select("heure", "rang", "nom_station", F.round("taux_moyen", 4).alias("taux_moyen"))
)

df_rank.show(48, truncate=False)


+-----+----+--------------------------------+----------+
|heure|rang|nom_station                     |taux_moyen|
+-----+----+--------------------------------+----------+
|0    |1   |Gare du Nord - Denain           |0.8257    |
|0    |2   |Place de Verdun                 |0.8197    |
|1    |1   |Place de Verdun                 |0.8646    |
|1    |2   |Gare du Nord - Denain           |0.8169    |
|2    |1   |Place de Verdun                 |0.8562    |
|2    |2   |Gare du Nord - Denain           |0.8235    |
|3    |1   |Place de Verdun                 |0.8522    |
|3    |2   |Gare du Nord - Denain           |0.8147    |
|4    |1   |Place de Verdun                 |0.845     |
|4    |2   |Gare du Nord - Denain           |0.8153    |
|5    |1   |Place de Verdun                 |0.8509    |
|5    |2   |Gare du Nord - Denain           |0.8214    |
|6    |1   |Place de Verdun                 |0.8231    |
|6    |2   |Gare du Nord - Denain           |0.7779    |
|7    |1   |Place de Verdun    

In [18]:
# Calcul de la variation du taux sur une heure glissante
# UNBOUNDED PRECEDING -> ligne actuelle = cumul depuis le début de la partition

# J'ai corrigé deux choses par rapport à l'énoncé de départ :
# - "station_id" n'existe pas dans notre schéma (voir printSchema plus haut),
#   c'est "nom_station" qui identifie une station de façon unique ;
# - je choisis une station réelle du jeu de données (celle qui ressort en tête
#   de la Question 3, la plus "pendulaire") plutôt qu'un id arbitraire.
fenetre_station_cumul = (
    Window
    .partitionBy("nom_station")
    .orderBy("horodatage")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

fenetre_station_ordre = Window.partitionBy("nom_station").orderBy("horodatage")

station_cible = "Gare du Nord - Denain"

# Variation par rapport au snapshot précédent (delta instantané),
# et moyenne cumulée depuis le début de l'historique de la station.
df_delta = (
    df
    .filter(F.col("nom_station") == station_cible)
    .withColumn("taux_precedent", F.lag("taux_occupation").over(fenetre_station_ordre))
    .withColumn("delta_instantane", F.round(F.col("taux_occupation") - F.col("taux_precedent"), 4))
    .withColumn("taux_cumule_moyen", F.round(F.avg("taux_occupation").over(fenetre_station_cumul), 4))
    .select("nom_station", "horodatage", "taux_occupation",
            "taux_precedent", "delta_instantane", "taux_cumule_moyen")
    .orderBy("horodatage")
)
df_delta.show(20, truncate=False)

+---------------------+-------------------+---------------+--------------+----------------+-----------------+
|nom_station          |horodatage         |taux_occupation|taux_precedent|delta_instantane|taux_cumule_moyen|
+---------------------+-------------------+---------------+--------------+----------------+-----------------+
|Gare du Nord - Denain|2020-11-26 12:59:00|0.05           |NULL          |NULL            |0.05             |
|Gare du Nord - Denain|2020-11-26 13:06:00|0.05           |0.05          |0.0             |0.05             |
|Gare du Nord - Denain|2020-11-26 13:21:00|0.05           |0.05          |0.0             |0.05             |
|Gare du Nord - Denain|2020-11-26 13:32:00|0.1            |0.05          |0.05            |0.0625           |
|Gare du Nord - Denain|2020-11-26 13:47:00|0.0            |0.1           |-0.1            |0.05             |
|Gare du Nord - Denain|2020-11-26 14:25:00|0.1            |0.0           |0.1             |0.0583           |
|Gare du N

---
## 1.4 Delta Lake : transactions, time-travel et MERGE

Delta Lake est une couche de stockage transactionnel construite par-dessus Parquet.
Elle apporte à Spark les propriétés **ACID** qui manquent au Parquet brut :

| Propriété | Parquet brut | Delta Lake |
|-----------|-------------|------------|
| Lecture cohérente pendant une écriture | Non | Oui (MVCC) |
| Annulation d'une écriture partielle | Non | Oui |
| Historique des versions | Non | Oui (time-travel) |
| Mise à jour / suppression de lignes | Non | Oui (MERGE, UPDATE, DELETE) |
| Optimisation automatique | Non | Oui (OPTIMIZE, Z-ORDER) |

### Écriture en format Delta


In [19]:
from delta.tables import DeltaTable

# Écriture initiale : toutes les données de 2020
# Adapté : nos données couvrent 2020-2021 au lieu de 2022-2023
df_2020 = df.filter(F.col("annee") == 2020)

t0 = time.perf_counter()
(
    df_2020
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("annee", "mois")
    .save(str(DELTA_DISPONIBLE))
)
print(f"Écriture 2020 : {time.perf_counter()-t0:.1f} s  --  {df_2020.count():,} lignes")

# Vérification de la structure Delta
import os
fichiers_delta = list(Path(DELTA_DISPONIBLE).rglob("*.parquet"))
log_delta      = list(Path(DELTA_DISPONIBLE / "_delta_log").glob("*.json"))
print(f"Fichiers Parquet : {len(fichiers_delta)}")
print(f"Entrées dans le transaction log : {len(log_delta)}")

Écriture 2020 : 4.8 s  --  3,338,002 lignes
Fichiers Parquet : 3
Entrées dans le transaction log : 1


In [20]:
# Ajout des données 2021 -- mode "append"
df_2021 = df.filter(F.col("annee") == 2021)

t0 = time.perf_counter()
(
    df_2021
    .write
    .format("delta")
    .mode("append")
    .save(str(DELTA_DISPONIBLE))
)
print(f"Ajout 2021 : {time.perf_counter()-t0:.1f} s  --  {df_2021.count():,} lignes")

# Historique des versions : chaque opération crée une nouvelle version
delta_table = DeltaTable.forPath(spark, str(DELTA_DISPONIBLE))
delta_table.history().select(
    "version", "timestamp", "operation", "operationParameters"
).show(truncate=False)

Ajout 2021 : 7.6 s  --  7,583,966 lignes
+-------+-----------------------+---------+----------------------------------------------------+
|version|timestamp              |operation|operationParameters                                 |
+-------+-----------------------+---------+----------------------------------------------------+
|1      |2026-08-31 18:24:40.668|WRITE    |{mode -> Append, partitionBy -> []}                 |
|0      |2026-08-31 18:24:32.776|WRITE    |{mode -> Overwrite, partitionBy -> ["annee","mois"]}|
+-------+-----------------------+---------+----------------------------------------------------+



### Time-travel : interroger une version passée

Delta Lake conserve toutes les versions de la table dans le transaction log.
On peut interroger n'importe quelle version passée avec la clause
`VERSION AS OF` ou `TIMESTAMP AS OF`.


In [21]:
# Lecture de la version 0 (uniquement les données 2022)
df_v0 = (
    spark.read
    .format("delta")
    .option("versionAsOf", 0)
    .load(str(DELTA_DISPONIBLE))
)
print(f"Version 0 (2022 uniquement) : {df_v0.count():,} lignes")

# Lecture de la version courante
df_current = spark.read.format("delta").load(str(DELTA_DISPONIBLE))
print(f"Version courante (2022+2023) : {df_current.count():,} lignes")

# Enregistrement comme vue SQL pour la suite
df_current.createOrReplaceTempView("disponibilite_delta")

Version 0 (2022 uniquement) : 3,338,002 lignes
Version courante (2022+2023) : 10,921,968 lignes


### `MERGE INTO` : mise à jour incrémentale

`MERGE INTO` est l'opération la plus puissante de Delta Lake. Elle permet de
**synchroniser** une table cible avec une table source en une seule passe :
insertions des nouvelles lignes, mises à jour des lignes existantes,
suppressions optionnelles.

Cas d'usage typique : arrivée quotidienne d'un nouveau batch de snapshots.


In [22]:
# Simulation : un nouveau batch arrive avec des corrections
# (quelques lignes modifiées + quelques nouvelles lignes)
from pyspark.sql.functions import lit, current_timestamp

# On prend 500 snapshots existants et on simule une correction du taux_occupation
df_corrections = (
    df_current
    .filter(F.col("mois") == 1)
    .limit(500)
    .withColumn("taux_occupation", F.round(F.col("taux_occupation") * 0.98, 4))
    .withColumn("source", lit("correction_batch"))
)

# Quelques nouvelles lignes fictives (snapshots manqués)
df_nouveaux = (
    df_current
    .filter(F.col("mois") == 1)
    .limit(50)
    .withColumn("horodatage",
        F.col("horodatage") + F.expr("INTERVAL 2 YEARS"))
    .withColumn("annee", lit(2024))
    .withColumn("source", lit("nouveau_batch"))
)

df_batch = df_corrections.union(df_nouveaux).drop("source")
print(f"Batch entrant : {df_batch.count()} lignes ({df_corrections.count()} corrections + {df_nouveaux.count()} nouvelles)")


Batch entrant : 550 lignes (500 corrections + 50 nouvelles)


In [23]:
# MERGE INTO : upsert (update + insert)
(
    delta_table.alias("cible")
    .merge(
        df_batch.alias("source"),
        # Condition de correspondance : même station, même horodatage
        # Adapté : nom_station au lieu de station_id (colonne inexistante dans nos données)
        "cible.nom_station = source.nom_station AND cible.horodatage = source.horodatage"
    )
    .whenMatchedUpdateAll()     # si correspondance : on écrase toutes les colonnes
    .whenNotMatchedInsertAll()  # si pas de correspondance : on insère
    .execute()
)

# Vérification : la table a une nouvelle version
delta_table.history().select(
    "version", "timestamp", "operation",
    "operationMetrics"
).show(5, truncate=False)

+-------+-----------------------+---------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp              |operation|operationMetrics                                                                                                                                                                                                                                              

In [24]:
delta_table.history().select(
    "version", "timestamp", "operation"
).show(truncate=False)

+-------+-----------------------+---------+
|version|timestamp              |operation|
+-------+-----------------------+---------+
|2      |2026-08-31 18:24:52.475|MERGE    |
|1      |2026-08-31 18:24:40.668|WRITE    |
|0      |2026-08-31 18:24:32.776|WRITE    |
+-------+-----------------------+---------+



In [25]:
# [EXERCICE]
# Le MERGE précédent a mis à jour des lignes existantes.
# Utilisez le time-travel pour comparer le taux_occupation moyen
# de janvier 2022 AVANT et APRÈS le merge.
#
# Indice : lisez la version 1 (avant merge) et la version courante,
# puis comparez avec une agrégation.
# ──────────────────────────────────────────────────────────────────────────

# Votre code ici :

# Adapté : janvier 2020 au lieu de janvier 2022
MOIS_CIBLE = 1
ANNEE_CIBLE = 2021    
VERSION_AVANT = 2   # Dernière version stable avant le MERGE (voir history)

# Lecture des deux versions
df_avant = (
    spark.read
    .format("delta")
    .option("versionAsOf", VERSION_AVANT)
    .load(str(DELTA_DISPONIBLE))
    .filter((F.col("annee") == ANNEE_CIBLE) & (F.col("mois") == MOIS_CIBLE))
)

df_apres = (
    spark.read
    .format("delta")
    .load(str(DELTA_DISPONIBLE))
    .filter((F.col("annee") == ANNEE_CIBLE) & (F.col("mois") == MOIS_CIBLE))
)

# Comparaison
print("=== Janvier 2020 ===")
print(f"Nombre de lignes AVANT merge : {df_avant.count():,}")
print(f"Nombre de lignes APRÈS merge : {df_apres.count():,}")

taux_avant = df_avant.agg(F.avg("taux_occupation")).collect()[0][0]
taux_apres = df_apres.agg(F.avg("taux_occupation")).collect()[0][0]

print(f"\nTaux moyen AVANT merge : {taux_avant:.6f}")
print(f"Taux moyen APRÈS merge : {taux_apres:.6f}")
print(f"Delta                  : {(taux_apres - taux_avant):.6f}")
if taux_avant:
    print(f"Variation relative     : {100*(taux_apres - taux_avant)/taux_avant:.4f}%")


=== Janvier 2020 ===
Nombre de lignes AVANT merge : 1,507,065
Nombre de lignes APRÈS merge : 1,507,065

Taux moyen AVANT merge : 0.375778
Taux moyen APRÈS merge : 0.375778
Delta                  : 0.000000
Variation relative     : 0.0000%
